# Homework 5: Deploying Machine Learning Models

## Setup
Use Python 3.11.15 and `uv`:

~~~
uv sync --locked
uv run python smoke_test.py
~~~

The smoke test loads the checked-in model and verifies that inference is deterministic. Check the artifact before using it:

~~~
sha256sum pipeline.bin
~~~

It must be:

~~~
1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin
~~~

In [2]:
import os
import hashlib

# Download the starter files, skipping any already present
base = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/homework/05-deployment"

files = [
    ".python-version", "Dockerfile", "feature_defaults.json", "model.py",
    "model_metadata.json", "pipeline.bin", "predict.py", "pyproject.toml",
    "q6_test.py", "smoke_test.py", "train.py", "uv.lock",
]

for f in files:
    if not os.path.exists(f):
        url = f"{base}/{f}"
        !wget -q {url} -O {f}
        print("Downloaded:", f)
    else:
        print("Already have:", f)

# Make sure uv is installed, and call it via Python so the notebook can always find it
import sys
%pip install -q uv
uv = f"{sys.executable} -m uv"

# Install the exact locked dependencies (quick no-op if already installed)
!{uv} sync --locked

# Test that the model loads and predicts consistently
!{uv} run python smoke_test.py

# Check the model file is genuine
expected = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
actual = hashlib.sha256(open("pipeline.bin", "rb").read()).hexdigest()
print("Checksum OK" if actual == expected else f"Checksum MISMATCH: {actual}")

# Keep the installed packages out of Git (only add the line once)
if not os.path.exists(".gitignore") or ".venv/" not in open(".gitignore").read():
    with open(".gitignore", "a") as g:
        g.write(".venv/\n")

Already have: .python-version
Already have: Dockerfile
Already have: feature_defaults.json
Already have: model.py
Already have: model_metadata.json
Already have: pipeline.bin
Already have: predict.py
Already have: pyproject.toml
Already have: q6_test.py
Already have: smoke_test.py
Already have: train.py
Already have: uv.lock

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 0.73ms
Prepared 18 packages in 763ms                                            
░░░░░░░░░░░░░░░░░░░░ [0/18] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intenti

## Question 1 - environment check
Run `uv --version` and record the version in your notes. This is a setup check, not a graded answer: your local `uv` version is not a property of the model.

In [3]:
import sys

uv = f"{sys.executable} -m uv"

version = !{uv} --version
print("Answer:", version[0])

Answer: uv 0.13.5 (x86_64-unknown-linux-gnu)


## Question 2 - locked dependency
Open `pyproject.toml` and `uv.lock`. Which Scikit-Learn version is part of the 2026 reference environment?

- `1.6.1`
- `1.7.2`
- `1.8.0`
- `2.0.0`

Do not regenerate the lockfile for the graded run; use `uv sync --locked`.

In [4]:
import tomllib

# What the project asks for (pyproject.toml)
with open("pyproject.toml", "rb") as f:
    pyproject = tomllib.load(f)

for dep in pyproject["project"]["dependencies"]:
    if dep.startswith("scikit-learn"):
        print("pyproject.toml:", dep)

# What actually gets installed (uv.lock)
with open("uv.lock", "rb") as f:
    lock = tomllib.load(f)

for package in lock["package"]:
    if package["name"] == "scikit-learn":
        print("uv.lock:", package["version"])
        print("Answer:", package["version"])

pyproject.toml: scikit-learn==1.7.2
uv.lock: 1.7.2
Answer: 1.7.2


## Question 3 - load the model
Write a Python script that loads `pipeline.bin` with `pickle` and calls `predict_proba` for this lead:

~~~
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
~~~

Report the probability of conversion rounded to three decimal places. The grader accepts an absolute error of at most `0.005`; do not choose the nearest answer from a list.

In [12]:
%%writefile q3_predict.py
import pickle

with open("pipeline.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

probability = pipeline.predict_proba([lead])[0, 1]
print("Answer: Probability of conversion =", round(probability, 3))

Overwriting q3_predict.py


In [13]:
import sys

uv = f"{sys.executable} -m uv"
!{uv} run python q3_predict.py

Answer: Probability of conversion = 0.533


## Question 4 - serve the model
Start the provided API:

~~~
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
~~~

The API has `GET /health` and `POST /predict`. Send this second lead:

~~~
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
~~~

Report `conversion_probability` rounded to three decimal places. The same `0.005` absolute tolerance applies. The response must contain a probability in `[0, 1]` and a boolean `conversion` field.

### Part 1: start the server (in a terminal)

The API has to keep running while requests are sent to it, so it runs in a
terminal rather than a notebook cell (a cell running a server would never finish).

Open a terminal (**Terminal → New Terminal**) and run:

~~~
cd /workspaces/machine-learning-zoomcamp-homework/05-deployment
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
~~~

- Wait for `Uvicorn running on http://0.0.0.0:9696`, then **leave that terminal running**.
- If it says `uv: command not found`, run `pip install uv` in the same terminal first.
- `predict:app` means "the object called `app` in `predict.py`". `uvicorn` is the web server that runs it.
- The API has two endpoints: `GET /health` and `POST /predict`.

**When finished:** press **Ctrl+C** in that terminal to stop the server. This must be
done before Question 6, because the Docker container also needs port 9696.

In [14]:
### Part 2: send the request
import json
from urllib.request import Request, urlopen

lead = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

request = Request(
    "http://localhost:9696/predict",
    data=json.dumps(lead).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

with urlopen(request) as response:
    result = json.load(response)

print(result)
print("Answer:", round(result["conversion_probability"], 3))

{'conversion_probability': 0.769799, 'conversion': True}
Answer: 0.77


## Container
The checked-in `Dockerfile` is the canonical 2026 container configuration. It uses Python `3.11.15-slim-bookworm`, `uv 0.10.11`, and the checked-in `uv.lock`. It copies the frozen artifact and starts the API on port `9696`.

Build and run it:

~~~
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
~~~

### Container setup (in a terminal)

These steps run in a terminal, not the notebook.

**1. Stop the Question 4 server first.** In the terminal running `uvicorn`, press **Ctrl+C**.
The container needs port 9696, and only one thing can use a port at a time.

**2. Check Docker is available:**

~~~
docker --version
~~~

**3. Build the image** (from inside `05-deployment`):

~~~
cd /workspaces/machine-learning-zoomcamp-homework/05-deployment
docker build -t zoomcamp-model:2026-hw5 .
~~~

- `-t` gives the image a name and tag so it can be referred to later.
- The `.` at the end means "use the Dockerfile in this folder". Don't miss it off.
- The first build takes a minute or two (downloads the base image and installs packages).

**4. Run the container:**

~~~
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
~~~

- Wait for `Uvicorn running on ...`, then **leave it running** for Question 6.
- `-p 9696:9696` connects port 9696 on the codespace to port 9696 inside the container.
- `--rm` deletes the container when it's stopped, so old containers don't pile up.
- Press **Ctrl+C** to stop it when finished.

### What the Dockerfile does

| Line | Meaning |
|---|---|
| `FROM python:3.11.15-slim-bookworm@sha256:...` | Start from a minimal Linux image with this exact Python version |
| `COPY --from=ghcr.io/astral-sh/uv:0.10.11...` | Copy in the `uv` tool, pinned to an exact version |
| `WORKDIR /code` | Work in a `/code` folder inside the container |
| `COPY pyproject.toml uv.lock ...` + `RUN uv sync --locked` | Install the exact locked packages (same as the Setup step) |
| `COPY model.py predict.py ... pipeline.bin ...` | Copy in the model and API code |
| `ENTRYPOINT ["uvicorn", "predict:app", ...]` | Start the server when the container runs (same command as Question 4) |

The container bundles everything from Setup and Question 4 (the right Python version, the
locked packages, the model and the server command), so it runs identically anywhere Docker
is installed. The `@sha256:...` parts pin the exact image contents, like the checksum on `pipeline.bin`.

## Question 5 - look at the container configuration
Which Python base-image tag is declared in the Dockerfile?

- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`

In [16]:
with open("Dockerfile") as f:
    for line in f:
        if line.strip().upper().startswith("FROM"):
            image = line.split()[1]      # the part after FROM
            tag = image.split("@")[0]    # drop the @sha256:... checksum
            print("Full line:", line.strip())
            print("Answer:", tag)

Full line: FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
Answer: python:3.11.15-slim-bookworm


## Question 6 - run the container
Run the same request from Question 4 against the container. You can use the provided smoke client:

~~~
python q6_test.py
~~~

Report `conversion_probability` rounded to three decimal places. The expected result is the same reference inference as Question 4, with the same `0.005` absolute tolerance. Also verify that:

~~~
curl -s http://localhost:9696/health
~~~

returns `{"status":"ok", ...}` and the model checksum recorded in `model_metadata.json`.

In [17]:
import ast
import json
from urllib.request import urlopen

# 1. Run the provided smoke client against the container
output = !python q6_test.py
print("q6_test.py output:", output[-1])

result = ast.literal_eval(output[-1])
print("Answer:", round(result["conversion_probability"], 3))

# 2. Check the health endpoint and the model checksum
health = json.load(urlopen("http://localhost:9696/health"))

with open("model_metadata.json") as f:
    metadata = json.load(f)

print("Status:", health["status"])
print("Checksum matches metadata:", health["model_sha256"] == metadata["model_sha256"])

q6_test.py output: {'conversion_probability': 0.769799, 'conversion': True}
Answer: 0.77
Status: ok
Checksum matches metadata: True


## Submit the results
- Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw05
- Submit numeric probabilities to three decimal places.
- Share what you learned in a short post using `#mlzoomcamp` and submit the post link with your homework.